In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("FastFood-P1-Cluster")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "jupyter")
         .config("spark.driver.bindAddress", "0.0.0.0")
         .config("spark.executor.memory", "512m")
         .getOrCreate())
sc = spark.sparkContext

sc.parallelize(range(1000)).count()   # force l'enregistrement des executors

print("Version Spark      :", spark.version)
print("Master             :", sc.master)
print("AppName            :", sc.appName)
print("defaultParallelism :", sc.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 22:24:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 2) / 2]

Version Spark      : 3.5.1
Master             : spark://spark-master:7077
AppName            : FastFood-P1-Cluster
defaultParallelism : 4


## Rôle des composants dans notre déploiement
- **Driver** : le processus Python/JVM lancé dans le conteneur `jupyter`. Il exécute le programme, construit le DAG, le découpe en stages et tasks, et les planifie.
- **Cluster Manager** : le Spark Master standalone (`spark-master:7077`). Il connaît les ressources des workers (4 cœurs, 4 Go) et alloue les executors à l'application.
- **Executors** : des JVM lancées sur `spark-worker-1` et `spark-worker-2`. Ils exécutent les tasks, gardent les données en cache et renvoient les résultats au driver.
- **SparkContext** : le point d'entrée créé par le driver (`spark.sparkContext`). Il ouvre la connexion au Cluster Manager, demande les executors et soumet les jobs.

## Deux limites de MapReduce corrigées par Spark
1. **Écriture disque entre chaque étape** : MapReduce matérialise les résultats intermédiaires sur HDFS. Spark garde les données en mémoire et chaîne les transformations dans un DAG. Dans ce projet, nettoyage → jointure → agrégation → PageRank (itératif, 10 itérations) s'exécutent sans relire le disque à chaque étape.
2. **Batch uniquement** : MapReduce ne traite pas les flux. Spark unifie batch et streaming avec la même API (Structured Streaming). Dans ce projet, le même code DataFrame sert pour l'historique (Parties 2 à 4) et pour le flux Kafka temps réel (Parties 5 et 6).

In [2]:
spark.stop()